# Scraper Pipeline
### Interview Questions — Multi-Source Pipeline
Run cells top to bottom for a full scrape. Each cell can also be re-run independently if something goes wrong mid-run.

In [ ]:
# Cell 1 — Imports and setup
import os
import sys
sys.path.insert(0, os.path.abspath(".."))

# Fix Windows terminal encoding so Unicode file paths don't crash
sys.stdout.reconfigure(encoding='utf-8')

from scraper.config import SOURCES, STAGING_DB, QUESTIONS_DB, ABANDON_MIN_FILES, ABANDON_EMPTY_PCT
from scraper.pipeline.store import (
    run_migrations, get_connection,
    save_question, save_incomplete, save_review, save_failed_page,
    get_all_hashes, get_all_embeddings,
    get_all_hashes_incomplete, get_all_embeddings_incomplete,
    touch_scraped_at,
)
from scraper.pipeline.discover import discover
from scraper.pipeline.crawl import crawl
from scraper.pipeline.extract import extract
from scraper.pipeline.dedup import process_question, hash_question
from scraper.pipeline.queue import CrawlQueue
from scraper.pipeline.validate import run_validation
from scraper.pipeline.promote import promote

# Initialize staging database
run_migrations(STAGING_DB)
conn = get_connection(STAGING_DB)
print(f"Staging DB ready: {STAGING_DB}")
print(f"Sources loaded: {[s['company'] for s in SOURCES]}")

In [ ]:
# Cell 2 — Discovery: find items for each source
all_items = []
for source in SOURCES:
    items = discover(source)
    all_items.extend(items)

print(f"\nTotal items to crawl: {len(all_items)}")

In [ ]:
# Cell 3 — Crawl + Extract + Dedup (full pipeline run)
queue = CrawlQueue()

# Load dedup caches once for each table
known_hashes = get_all_hashes(conn)
known_embeddings = get_all_embeddings(conn)
known_hashes_incomplete = get_all_hashes_incomplete(conn)
known_embeddings_incomplete = get_all_embeddings_incomplete(conn)

stats = {
    "items_scanned": 0,
    "files_processed": 0,
    "files_skipped": 0,
    "questions_extracted": 0,
    "added_complete": 0,
    "added_incomplete": 0,
    "dates_refreshed": 0,
    "variants_linked": 0,
    "sent_to_review": 0,
    "failed_pages": 0,
    "urls_discovered": 0,
    "items_abandoned": 0,
}


def _item_label(item: dict) -> str:
    """Human-readable label for any source item type."""
    t = item.get("type")
    if t == "github_repo":
        return f"{item.get('full_name', '?')} ({item.get('stars', '-')} stars)"
    if t == "reddit_post":
        return f"r/{item.get('subreddit', '?')}: \"{item.get('title', '')[:70]}\""
    return item.get("url", item.get("raw_url", "?"))


def process_file(file: dict):
    url = file.get("raw_url") or file.get("url")
    try:
        result = extract(file, conn=conn)
    except (ConnectionError, RuntimeError) as e:
        save_failed_page(conn, url, str(e))
        stats["failed_pages"] += 1
        print(f"    FAILED ({e})", flush=True)
        return False

    # File-level dedup: content unchanged since last scrape
    if result.get("file_skipped"):
        stats["files_skipped"] += 1
        return False

    questions = result["questions"]
    discovered_urls = result["discovered_urls"]

    if discovered_urls:
        queue.add_many(discovered_urls, file.get("company", "Unknown"), source="discovered")
        stats["urls_discovered"] += len(discovered_urls)

    if not questions:
        print(f"    0 questions (skipped)", flush=True)
        return False

    stats["files_processed"] += 1
    stats["questions_extracted"] += len(questions)

    for q in questions:
        if q.get("confidence") == "uncertain":
            save_review(conn, q, q.get("reasoning", ""))
            stats["sent_to_review"] += 1
            continue

        is_complete = bool(q.get("company") and q.get("role"))
        q_hash = hash_question(q["question_text"])

        if is_complete:
            if q_hash in known_hashes:
                touch_scraped_at(conn, q_hash)
                stats["dates_refreshed"] += 1
                continue
            deduped = process_question(q, known_hashes, known_embeddings)
            if deduped is None:
                continue
            is_variant = deduped.get("canonical_id") is not None
            qid = save_question(conn, deduped, deduped.get("canonical_id"))
            known_hashes.add(deduped["content_hash"])
            known_embeddings.append({"id": qid, "canonical_id": deduped.get("canonical_id") or qid, "embedding": deduped["embedding"]})
            stats["added_complete"] += 1
            if is_variant:
                stats["variants_linked"] += 1
        else:
            if q_hash in known_hashes_incomplete:
                touch_scraped_at(conn, q_hash)
                stats["dates_refreshed"] += 1
                continue
            deduped = process_question(q, known_hashes_incomplete, known_embeddings_incomplete)
            if deduped is None:
                continue
            qid = save_incomplete(conn, deduped, deduped.get("canonical_id"))
            known_hashes_incomplete.add(deduped["content_hash"])
            known_embeddings_incomplete.append({"id": qid, "canonical_id": deduped.get("canonical_id") or qid, "embedding": deduped["embedding"]})
            stats["added_incomplete"] += 1

    print(f"    {len(questions)} questions extracted, {stats['added_complete']} complete / {stats['added_incomplete']} incomplete in bank", flush=True)
    return True


# --- Phase 1: process all discovered items ---
for i, item in enumerate(all_items):
    print(f"\n[{item['company']}] Item {i+1}/{len(all_items)}: {_item_label(item)}", flush=True)
    stats["items_scanned"] += 1
    files = crawl(item)
    print(f"  Found {len(files)} file(s)", flush=True)
    item_empty = 0
    item_total = 0
    for file in files:
        print(f"  Processing: {file['path']}", flush=True)
        found = process_file(file)
        item_total += 1
        if not found:
            item_empty += 1
        # Auto-abandon only applies to multi-file items (GitHub repos)
        if item.get("type") == "github_repo" and item_total >= ABANDON_MIN_FILES and (item_empty / item_total) >= ABANDON_EMPTY_PCT:
            remaining = len(files) - item_total
            print(f"  Abandoning repo — {item_empty}/{item_total} files empty ({remaining} files skipped)", flush=True)
            stats["items_abandoned"] += 1
            break

# --- Phase 2: follow discovered URLs ---
if not queue.is_empty():
    print(f"\n--- Following {queue.size()} discovered URLs ---", flush=True)
    while not queue.is_empty():
        item = queue.pop()
        print(f"\n  URL: {item['url']}", flush=True)
        file = {"type": "url", "raw_url": item["url"], "url": item["url"], "path": item["url"], "repo": "discovered", "company": item["company"]}
        process_file(file)

# --- Summary ---
print("\n" + "-" * 50)
print("Scrape complete")
print(f"  Items scanned:        {stats['items_scanned']}")
print(f"  Items abandoned:      {stats['items_abandoned']}")
print(f"  Files skipped:        {stats['files_skipped']} (unchanged since last scrape)")
print(f"  Files processed:      {stats['files_processed']}")
print(f"  Questions extracted:  {stats['questions_extracted']}")
print(f"  Complete (co+role):   {stats['added_complete']}")
print(f"  Incomplete:           {stats['added_incomplete']}")
print(f"  Dates refreshed:      {stats['dates_refreshed']}")
print(f"  Variants linked:      {stats['variants_linked']}")
print(f"  Sent to review:       {stats['sent_to_review']}")
print(f"  URLs discovered:      {stats['urls_discovered']}")
print(f"  Failed pages:         {stats['failed_pages']}")
print("-" * 50)

In [ ]:
# Cell 4 — Validation report
# Review this before running Cell 5
run_validation(STAGING_DB)

In [ ]:
# Cell 5 — Promote to production
# Only run this after reviewing the validation report above
promote(STAGING_DB, QUESTIONS_DB)